# Reality2Unity — Week 1 Baseline (Colab)

Google Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서부터 실행하세요.
기존 `python/main.py`의 `main()`을 호출하여 DPT_Large 추론 → 정규화된 Depth Map → Top-view Heightmap → 3D 시각화를 실행합니다. 계산이나 변환 알고리즘은 노트북에 복사하지 않습니다.

**먼저 현재 모듈과 이 노트북을 GitHub의 `feat/legacy-baseline` 브랜치에 반영해야 합니다.** 아직 로컬에만 있는 변경은 Colab에서 clone할 수 없습니다. 아래 브랜치는 모듈이 포함된 브랜치로 변경할 수 있습니다.

모델·가중치는 최초 실행 시 다운로드됩니다. 원본 해상도와 삼중 반복문을 그대로 사용하므로 큰 이미지는 변환 시간이 오래 걸릴 수 있습니다.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/mir0173/Reality2Unity.git"
BRANCH = "feat/legacy-baseline"
REPO_DIR = Path("/content/Reality2Unity")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"기존 체크아웃을 사용합니다: {REPO_DIR}")

required = [
    "python/main.py",
    "python/depth_estimation/midas.py",
    "python/terrain_generation/legacy_converter.py",
]
missing = [p for p in required if not (REPO_DIR / p).is_file()]
if missing:
    raise FileNotFoundError(f"모듈이 포함된 브랜치를 먼저 GitHub에 반영하세요: {missing}")


## 실행 패키지 준비

Colab의 PyTorch/torchvision을 사용하고 나머지 패키지를 런타임에 설치합니다. 저장소의 `requirements.txt`는 읽거나 수정하지 않습니다.


In [ ]:
%pip -q install timm opencv-python-headless matplotlib


In [ ]:
%matplotlib inline
import sys

python_dir = str(REPO_DIR / "python")
if python_dir not in sys.path:
    sys.path.insert(0, python_dir)

from main import main


## 입력 이미지 업로드

이전에 실험한 것과 동일한 이미지를 한 장 선택하세요. 업로드 파일은 Colab 런타임의 `/content/baseline_inputs/`에만 저장됩니다.


In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("비교할 이미지 한 장만 업로드하세요.")

filename, image_bytes = next(iter(uploaded.items()))
input_dir = Path("/content/baseline_inputs")
input_dir.mkdir(parents=True, exist_ok=True)
image_path = input_dir / Path(filename).name
image_path.write_bytes(image_bytes)
del uploaded, image_bytes


## Baseline 실행

기존 `main()`이 모델 로딩과 추론을 한 번 수행하고 아래 순서로 그림을 표시합니다.

1. 원본 이미지와 정규화·반전된 Depth Map (`Normalized Height Map`)
2. 원본 이미지와 Top-view Heightmap
3. Top-view Heightmap의 3D 표면

원본 함수의 정규화·깊이 반전·변환 순서 및 컬러맵을 그대로 사용합니다.


In [ ]:
main(str(image_path))


## 저장 전 출력 정리

이 노트북은 실행 횟수와 출력이 비어 있는 상태로 제공됩니다. 실행 후 공유하거나 Git에 반영하기 전에는 Colab의 **수정 → 모든 출력 지우기**를 실행하세요. Colab 설정에서 **저장 시 코드 셀 출력 생략**도 활성화하세요. 업로드 이미지와 모델 가중치는 저장소에 추가하지 마세요.
